# Reference · Day 4 studio — the size of a leak is a number

**Not a marking key.** You triaged eight workflows and measured two of them. Your
numbers will not match these — you picked a different pair and different seeds. What
is worth comparing is the *shape*: where each workflow lands on the one axis that
actually matters.

The room arrived believing leakage is a property a workflow either has or does not
have. You leave knowing it is a **magnitude** — and that the magnitude is something
you *measure*, not something you argue about. The clearest evidence is that two of
today's workflows are the **identical structural mistake** and sit at opposite ends
of the table below: one costs `0.000000`, the other builds a model out of pure noise.
Telling them apart took a number, not an opinion.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")

import numpy as np
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import (GroupKFold, KFold, StratifiedKFold,
                                     TimeSeriesSplit, cross_val_score)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from stat764 import load

ames = load("ames.csv")
bike = load("bike_hour.csv")
print(f"{len(ames):,} houses · {len(bike):,} hours of bike rentals")

## One measurement, reused

Part B gave you the pattern three times: **build the leaky version, build the honest
version, report the gap.** Everything below is that one sentence applied to workflow
after workflow. Writing the comparison *once* and swapping the workflow in is the
whole discipline — if each leak were measured by its own block of copied code, a
difference between two leaks could be a difference in the code.

`leaks` collects each result as `(leaky, honest, metric)` so the spectrum at the end
can line them all up on the same axis.

In [ ]:
leaks = {}   # label -> (leaky score, honest score, metric name)


def record(label, leaky, honest, metric="R2"):
    leaks[label] = (leaky, honest, metric)
    print(f"  {label:<34} leaky {leaky:+.6f}   honest {honest:+.6f}   gap {leaky - honest:+.6f}")

## #1 · Standardize on all the data, then split — *the zero*

The textbook leak: the scaler learns its mean and SD from every row, test folds
included, before the split. Real in principle. So measure it — across three model
families, because the answer depends on the model.

In [ ]:
cols = ["Gr_Liv_Area", "Lot_Area", "Year_Built", "Overall_Qual", "Total_Bsmt_SF"]
Xa = ames[cols].fillna(ames[cols].median())
ya = ames["SalePrice"]
kf = KFold(5, shuffle=True, random_state=764)

for name, mk in [("OLS", LinearRegression),
                 ("Ridge(100)", lambda: Ridge(alpha=100)),
                 ("kNN(5)", lambda: KNeighborsRegressor(5))]:
    leaky = cross_val_score(mk(), StandardScaler().fit_transform(Xa), ya,
                            cv=kf, scoring="r2").mean()
    honest = cross_val_score(Pipeline([("s", StandardScaler()), ("m", mk())]), Xa, ya,
                             cv=kf, scoring="r2").mean()
    record(f"#1 standardize · {name}", leaky, honest)

For **OLS the gap is exactly `0.000000`, and always will be** — least squares is
equivariant to affine rescaling of the predictors, so restandardizing cannot move a
single fitted value. The leak is real but has no channel through which to act. Ridge
and kNN genuinely care about scale, so their gap is nonzero — and still in the
*fourth decimal*, because a mean over ~2,900 rows barely shifts when you drop a fifth
of them. Real, negligible, and model-dependent, all at once.

## #3 · Impute with the full-data median, then CV — same mechanism

A parameter (the median) estimated from every row, test folds included — structurally
identical to #1. Inject 15% missingness and measure it against imputing *inside* each
fold.

In [ ]:
rng = np.random.default_rng(764)
Xm = ames[cols].astype(float).copy()
Xm = Xm.mask(rng.random(Xm.shape) < 0.15)          # blank ~15% at random

leaky = cross_val_score(LinearRegression(), Xm.fillna(Xm.median()), ya,
                        cv=kf, scoring="r2").mean()
honest = cross_val_score(Pipeline([("i", SimpleImputer(strategy="median")),
                                   ("m", LinearRegression())]), Xm, ya,
                         cv=kf, scoring="r2").mean()
record("#3 impute full-median", leaky, honest)

The gap is in the *fifth* decimal and can even land slightly negative — at 15% missing
on 2,930 rows the full-data median and the per-fold median are almost the same number,
so the contamination has nothing to bite on. It grows with the missing fraction; here
it is negligible. A team that answered "**DEPENDS** — on how much is missing" saw
further than one that answered "LEAK."

## #4 · Screen 5,000 → 20 on all the data, then CV — *the disaster*

Now the same structural mistake as #1, on data with **no signal at all**: `y` is drawn
independently of `X`, so the true predictive R² is exactly zero. We screen to the 20
columns most correlated with the outcome — looking at every row, test folds included —
then cross-validate honestly.

In [ ]:
rng = np.random.default_rng(764)
n, p = 200, 5000
Xn = rng.normal(size=(n, p))
yn = rng.normal(size=n)                             # independent of Xn, by construction
k0 = KFold(5, shuffle=True, random_state=0)

corr = np.abs([np.corrcoef(Xn[:, j], yn)[0, 1] for j in range(p)])
top20 = np.argsort(corr)[-20:]
leaky = cross_val_score(Ridge(), Xn[:, top20], yn, cv=k0, scoring="r2").mean()
honest = cross_val_score(Pipeline([("s", SelectKBest(f_regression, k=20)), ("m", Ridge())]),
                         Xn, yn, cv=k0, scoring="r2").mean()
record("#4 screen 5000->20 on noise", leaky, honest)

**R² ≈ 0.435, out of data that contains nothing.** The screen kept the 20 columns that
happened to correlate with the outcomes — including the outcomes of rows that later
served as test folds — so those coincidences were still sitting in the test folds when
the model was scored. Put the screen *inside* the pipeline and it is refit per fold,
never seeing what it will be graded on: the honest number is negative, which is the
correct verdict for noise. **Same mistake as #1. Five orders of magnitude more damage.**

## #5 · Oversample to balance, then CV — the resampling myth

The outcome is a rare "peak demand" hour (~5%). The common advice is to balance the
classes first, then cross-validate the balanced dataset. Metric here is AUC, not R².

In [ ]:
peak = (bike["cnt"] > bike["cnt"].quantile(0.95)).astype(int)
X_peak = bike[["hr", "temp", "atemp", "hum", "windspeed", "workingday",
               "weathersit", "season", "yr", "mnth", "holiday", "weekday"]]


def oversample(X_part, y_part, seed=0):
    """Duplicate minority rows at random until the classes are balanced."""
    r = np.random.default_rng(seed)
    minority = np.where(y_part == 1)[0]
    majority = np.where(y_part == 0)[0]
    extra = r.choice(minority, size=len(majority) - len(minority), replace=True)
    keep = np.concatenate([majority, minority, extra])
    return X_part.iloc[keep], y_part.iloc[keep]


X_bal, y_bal = oversample(X_peak, peak)
myth = cross_val_score(KNeighborsClassifier(1), X_bal, y_bal,
                       cv=StratifiedKFold(5, shuffle=True, random_state=0),
                       scoring="roc_auc").mean()
aucs = []
for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(X_peak, peak):
    X_tr, y_tr = oversample(X_peak.iloc[tr], peak.iloc[tr])
    fit = KNeighborsClassifier(1).fit(X_tr, y_tr)
    aucs.append(roc_auc_score(peak.iloc[te], fit.predict_proba(X_peak.iloc[te])[:, 1]))
record("#5 oversample, then CV", myth, np.mean(aucs), metric="AUC")

Oversampling makes **copies of rows**; random folds then scatter a row and its own
duplicate into different folds, and a 1-nearest-neighbor classifier answers a test row
by finding its exact copy in the training fold. It is not learning about peak demand —
it is recognizing rows it has already seen. Balance *inside* each training fold and the
inflation goes away. (SMOTE is not exempt: its synthetic points are interpolations of
real ones, so they are not independent of the rows they were built from.)

## The spectrum — one axis, everything on it

This table is the whole studio. Same measurement, six times, sorted by how much the
leak inflated the estimate. Read the range, not any single row.

In [ ]:
print(f"  {'workflow':<34}{'metric':>7}{'leaky':>10}{'honest':>10}{'gap':>11}")
for label, (lk, hon, metric) in sorted(leaks.items(), key=lambda kv: kv[1][0] - kv[1][1]):
    print(f"  {label:<34}{metric:>7}{lk:>10.4f}{hon:>10.4f}{lk - hon:>+11.6f}")

## The two questions

Look at the top and bottom rows. **#1 (standardize, OLS)** and **#4 (screen on noise)**
are the *same structural error* — a parameter estimated on all the rows, test folds
included, before the split. One costs `0.000000`. The other conjures R² = 0.435 out of
nothing.

That is the lesson of the day: **"is it leakage?" and "does it matter?" are two
different questions, and only the second one has a number attached.** The first you
answer by thinking about mechanism; the second you answer only by measuring. Follow the
rule for *both* — #1 costs nothing to fix, and the identical slip in #4 is
catastrophic — but never confuse having named a leak with having measured one.

## #2 · The clean one — the argument you were supposed to have

Log-transforming a right-skewed outcome *before* the split is **not a leak**, and it is
the item designed to start a fight. `log()` is a fixed function with **no parameters
estimated from data** — nothing about the test rows changes what happens to the
training rows, so there is no information to cross the split.

"Never touch the data before splitting" is a slogan standing in for the real rule:
*don't let anything learned from the test rows reach the model.* A parameterless
transform has nothing to learn. Contrast #1 directly — the scaler's mean and SD **are**
learned from data, which is exactly why standardizing *can* leak and taking a log
*cannot*. If your team split on #2, that is the right instinct meeting the real
boundary of the rule.

## #6 · Report the best of 200 CV scores — the subtle one

Nothing crosses the train/test line here, so it feels safe. `y` is again pure noise.
We try 200 candidate models (each a random handful of columns), cross-validate each one
*honestly*, and report the best score we saw.

In [ ]:
rng = np.random.default_rng(6)
n, p = 200, 100
Xw = rng.normal(size=(n, p))
yw = rng.normal(size=n)                             # no signal
cvw = KFold(5, shuffle=True, random_state=1)

scores = np.array([cross_val_score(LinearRegression(), Xw[:, rng.choice(p, 5, replace=False)],
                                   yw, cv=cvw, scoring="r2").mean() for _ in range(200)])
print(f"  best of 200 honest CV scores: {scores.max():+.3f}")
print(f"  mean of the 200:              {scores.mean():+.3f}   (the truth is 0)")

Every one of the 200 estimates was honestly cross-validated, yet the **maximum** of 200
noisy estimates is a biased estimate of the performance of the model that achieved it —
you selected the luckiest split-and-columns combination, and luck does not repeat on new
data. This is the **winner's curse**, and it is why *nested* cross-validation exists: an
outer loop around the entire "try 200, pick the best" procedure. We do it properly on
**October 1**. "But I did cross-validate" is exactly the confusion that meeting exists
to fix.

## #7 and #8 · The leaks that live in the *split*, not the data

The last two cannot be caught by re-reading column names, because nothing about the
columns is wrong — the flaw is in **how you split**. These are the two that will bite
the capstone.

**#7 — several encounters per patient, split at random.** Give each patient a constant
"fingerprint" (features that repeat across their encounters) and an outcome that depends
only on *who they are*. A random split drops other encounters of the same patient into
the training fold; the model recognizes the patient and answers perfectly. Hold whole
patients out with `GroupKFold` and the fingerprint is worthless.

In [ ]:
rng = np.random.default_rng(7)
n_pat = 200
enc = rng.integers(2, 6, size=n_pat)                # encounters per patient
pid = np.repeat(np.arange(n_pat), enc)
N = len(pid)
fingerprint = np.repeat(rng.normal(size=(n_pat, 5)), enc, axis=0) + rng.normal(scale=0.01, size=(N, 5))
y_pat = np.repeat(rng.normal(size=n_pat), enc) + rng.normal(scale=0.1, size=N)

random_split = cross_val_score(KNeighborsRegressor(5), fingerprint, y_pat,
                               cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()
grouped = cross_val_score(KNeighborsRegressor(5), fingerprint, y_pat, groups=pid,
                          cv=GroupKFold(5), scoring="r2").mean()
print(f"  #7 random row split: R2 {random_split:+.3f}     GroupKFold on patient: R2 {grouped:+.3f}")

**#8 — predict the future, split the quarters at random.** A random split lets the model
*interpolate* between past and future points; the real task is to *forecast* past the
end of what it has seen. On a random walk, random K-fold interpolates its neighbors and
looks excellent; a forward, time-ordered split has to extrapolate and cannot.

In [ ]:
rng = np.random.default_rng(8)
Nt = 400
t = np.arange(Nt)[:, None]
walk = np.cumsum(rng.normal(size=Nt))               # nonstationary series

random_time = cross_val_score(KNeighborsRegressor(5), t, walk,
                              cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()
forward = cross_val_score(KNeighborsRegressor(5), t, walk,
                          cv=TimeSeriesSplit(5), scoring="r2").mean()
print(f"  #8 random split: R2 {random_time:+.3f}     forward (time-ordered) split: R2 {forward:+.3f}")

Both fixes are choices of **splitter**, not edits to the features: `GroupKFold` on the
patient id, a time-ordered split on the date. You meet #7 again on **October 6** with
Diabetes-130 (the most common flaw in public notebooks on that dataset) and #8 on
**September 24** with the Ames assessment dates, and again on **October 22** when
temporal structure breaks the exchangeability that conformal prediction assumes. When
you choose a dataset for the capstone, ask first: *does a row repeat an entity, and does
time run through it?*

## Check your own triage against this

Not "did I get these numbers." These:

| | |
|---|---|
| **1** | For every workflow you called a leak, did you name **what information reaches the model that would not exist at prediction time** — not just "it leaks"? |
| **2** | For the two you measured, did you build the honest version with the **same code**, changing only where the suspect step happens? A leak's size is a difference; both sides must be measured identically. |
| **3** | Did you separate **"is it leakage?"** from **"does it matter?"** — and did you resist calling #1 large just because it is real, or #2 a leak just because it happens before the split? |
| **4** | Could you say, for #6, why a cross-validated number can still be biased when nothing crossed the train/test line? |
| **5** | For a capstone dataset, can you say whether it has the **#7 (repeated entity)** or **#8 (time)** structure — before you split it? |

⚠ **You arrived thinking leakage was a yes/no property of a workflow. The one number to
leave with is the *range* of the spectrum above** — from `0.000000` to a model built out
of noise — and the habit of finding out which end you are on by measuring, not arguing.